# 05. 전체 그래프 조립 + 실행

01~04번 노트북에서 각자 저장한 `src/nodes_b.py`, `src/nodes_cd.py`, `src/nodes_e.py`, `src/nodes_fgh.py`를 가져와
`A -> B -> {C,D,E} -> F -> G <-> H` 그래프로 조립하고 실제로 돌린다.

**먼저 01~04번 노트북을 각자 한 번씩 끝까지 실행해서 파일을 생성해둬야 한다.** 넷 중 하나라도 안 돌렸으면 이 노트북의 import가 실패한다.

In [ ]:
import sys
sys.path.insert(0, "..")

import os
from dotenv import load_dotenv

# override=True 가 핵심이다. load_dotenv 는 기본적으로 "이미 있는" 환경변수를
# 덮어쓰지 않는다. 셸에 낡은 OPENAI_API_KEY 가 export 돼 있으면 .env 의 진짜
# 키가 무시되고 401 이 난다 - 2026-09-22 에 실제로 여기서 막혔다.
load_dotenv("../.env", override=True)

# 키 값은 절대 찍지 않는다. 있는지와 길이만 본다.
for key in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    v = os.environ.get(key)
    print(f"{key}: {str(len(v)) + '자' if v else '🔴 없음 - 아래 실제 실행 셀은 실패한다'}")

## 1. 각 담당 노트북이 만든 노드 함수 가져오기

In [ ]:
from src.nodes_b import make_node_b
from src.nodes_cd import make_node_c, make_node_d
from src.nodes_e import make_node_e
from src.nodes_fgh import make_node_f, make_node_g, node_h_validate, route_after_h

print("네 파일 전부 import 성공")

## 2. A. 기술 선정 (Human 기반, 정적)

LLM 호출이 없어서 별도 노트북 없이 여기서 바로 정의한다.

In [ ]:
import inspect
from src.graph import node_a_select_technologies

# A 는 LLM 호출이 없는 정적 노드라 src/graph.py 에 바로 들어 있다.
# 여기서 다시 정의하면 graph.py 와 갈라지므로 가져다 보기만 한다.
print(inspect.getsource(node_a_select_technologies))
print(node_a_select_technologies({}))

## 3. 리트리버 · LLM · 웹서치 도구 준비

01번·03번 노트북에서 이미 만들어본 것과 같다 - 여기서 다시 한번 만든다(프로세스가 다르므로).

In [ ]:
from src.ingest import build_tech_retriever, build_domain_retriever

print("리트리버 구축 중 (Qwen3-Embedding-0.6B 다운로드가 처음엔 걸릴 수 있음)...")
tech_retriever = build_tech_retriever()
domain_retriever = build_domain_retriever()
print("완료")

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch

# 모델 두 개를 쓴다 (src/graph.py 의 build_graph 가 둘 다 받는다).
#   llm      = gpt-4.1-mini : C, D, F - 단순 구조화 출력
#   llm_full = gpt-4.1      : B, E, G - 품질이 중요한 노드
llm = init_chat_model(
    config.LLM_MODEL, model_provider=config.LLM_PROVIDER, temperature=config.LLM_TEMPERATURE
)
llm_full = init_chat_model(
    config.LLM_MODEL_FULL, model_provider=config.LLM_PROVIDER, temperature=config.LLM_TEMPERATURE
)
web_search_tool = TavilySearch(max_results=5)
print(f"llm={config.LLM_MODEL} / llm_full={config.LLM_MODEL_FULL}")

## 4. 그래프 조립

3-4절 mermaid 그대로. Reducer가 붙은 State 키는 하나도 없다 — 모든 키를 쓰는 노드가 정확히 1개씩이라 동시 쓰기 충돌 지점이 없기 때문(3-1절).

In [ ]:
import inspect
from src.graph import build_graph

# 조립은 src/graph.py 에 한 벌만 둔다. 예전엔 이 셀이 build_graph 를 따로
# 정의하고 있어서, graph.py 가 llm_full 을 받도록 바뀐 뒤에도 노트북만 4인자
# 옛 판으로 남아 있었다. 가져다 쓰면 갈라질 수가 없다.
print(inspect.getsource(build_graph))

graph = build_graph(llm, llm_full, tech_retriever, domain_retriever, web_search_tool)
print("그래프 컴파일 완료")

## 5. 그래프 구조 확인 (mermaid)

design doc 3-4절과 비교해서 다른 점이 없는지 눈으로 확인.

In [ ]:
print(graph.get_graph().draw_mermaid())

## 6. 배선 재확인 — API 키 없이 (선택)

01~04번에서 각자 검증했지만, 넷을 실제로 이어붙였을 때도 문제없는지 한 번 더 가짜 객체로 전체를 돌려본다.

In [ ]:
"""API 키 없이 그래프 배선(엣지·State 병합·H 루프)만 검증한다.

원래 tests/test_graph_wiring.py 에 있던 것을, 그 파일이 지워져서(9bd91e0)
노트북 안으로 옮겨 왔다. 가짜 객체는 실제 LangChain 객체의 인터페이스
(.invoke / .with_structured_output)만 흉내낸다.
"""
from langchain_core.documents import Document

from src import config
from src.graph import build_graph
from src.schemas import (
    DomainCriteria, DomainEval, MarketEval, ResearchResult,
    StakeholderEval, TechResearch, TechStatus, TRLAssessment,
)


class FakeStructuredLLM:
    def __init__(self, output):
        self.output = output
    def invoke(self, prompt):
        return self.output


class FakeMessage:
    def __init__(self, content):
        self.content = content


def good_report(with_domain=True, formatted=True):
    """H 의 검사 항목(장 유무 / 개조식 / [소결] / 번호 인용)을 골라 맞춘 보고서."""
    titles = ["시장", "이해관계자"] + (["도메인"] if with_domain else [])
    out = ["# SUMMARY", "내용"]
    for t in titles:
        out.append(f"# {t}")
        if formatted:
            out += [f"- 항목{i}: TurboQuant / InfiniGen 나란히 기록" for i in range(1, 6)]
            out.append(f"[소결] {t} 관점은 조건에 따라 갈린다.")
        else:
            out.append("줄글 문단이라 개조식이 아니다.")
    out.append("# REFERENCE")
    if formatted:
        out += [f'[{i}] 저자{i}, "제목{i}," 학회, 2024. https://example.com/{i}'
                for i in range(1, 6)]
    else:
        out.append("URL 나열")
    return "\n".join(out)


class FakeLLM:
    """TechStatus 는 제네릭이라 필드마다 허용 값이 다르다(schemas.py).
    한 벌을 돌려 쓰면 ValidationError 가 난다 - 필드별로 맞는 값을 넣는다."""

    _n = {"report": 0}

    def with_structured_output(self, schema_cls):
        if schema_cls is TechResearch:
            rr = ResearchResult(
                overview="가짜 개요", scope="가짜 범위", limitations="가짜 한계",
                trl_assessment=TRLAssessment(trl_ondevice=4, trl_global=6,
                                             evidence_status="found"),
            )
            return FakeStructuredLLM(TechResearch(turboquant=rr, infinigen=rr))
        if schema_cls is MarketEval:
            return FakeStructuredLLM(MarketEval(
                market_size_growth="추정 갈림",
                adoption_status=TechStatus(turboquant="정식", infinigen="실험"),
                ecosystem_support=TechStatus(turboquant="본류 병합", infinigen="포크만"),
                standardization="있음", label="조건 의존", notes="가짜",
            ))
        if schema_cls is StakeholderEval:
            return FakeStructuredLLM(StakeholderEval(
                competing_camp_reaction=TechStatus(turboquant="한계 지적", infinigen="언급만"),
                developer_adoption=TechStatus(turboquant="채택했다고 말함", infinigen="조건부"),
                investor_coverage=TechStatus(turboquant="있음", infinigen="근거 없음"),
                label="조건 의존", notes="가짜",
            ))
        if schema_cls is DomainEval:
            return FakeStructuredLLM(DomainEval(
                criteria=DomainCriteria(
                    memory_budget=TechStatus(turboquant="들어간다 (1.2GB)",
                                             infinigen="넘는다 (1.8GB)"),
                    accuracy=TechStatus(turboquant="나눠 보고 (-2%p)",
                                        infinigen="뭉쳐 보고 (-4%p)"),
                    latency=TechStatus(turboquant="이 조건 실측 (45ms)",
                                       infinigen="다른 조건 실측 (30ms)"),
                    power_thermal=TechStatus(turboquant="실측 있음 (3.2W)",
                                             infinigen="근거 없음"),
                ),
                reversal_detected=True, reversal_criteria="메모리 예산",
                label="조건 의존", notes="메모리 예산 - 가짜 근거.",
            ))
        # F 는 schemas.Synthesis 대신 create_model 로 만든 고정 필드 스키마를
        # 쓴다(Synthesis.labels 가 dict[str,str] 이라 OpenAI 구조화 출력이 거부).
        # 받은 schema_cls 를 그대로 인스턴스화하면 양쪽 다 통한다.
        if schema_cls.__name__ in ("Synthesis", "SynthesisStrict"):
            return FakeStructuredLLM(schema_cls(
                labels={"market": "조건 의존", "stakeholder": "조건 의존",
                        "domain": "조건 의존", "trl": "판단보류"},
                conflicts=["시장은 압축 쪽, 도메인은 확장 쪽을 일부 지지"],
                reasoning="도메인은 메모리 예산 기준으로 작동점 A/B 판정이 갈림.",
            ))
        raise ValueError(f"예상 못 한 스키마: {schema_cls}")

    def invoke(self, prompt):
        # G 용. 첫 판은 '도메인' 장 누락 + 형식 위반 -> H->G 재시도 루프를 태운다.
        self._n["report"] += 1
        first = self._n["report"] == 1
        return FakeMessage(good_report(with_domain=not first, formatted=not first))


class FakeRetriever:
    def invoke(self, query):
        return [Document(page_content=f"가짜 문서 for '{query}'",
                         metadata={"source_name": "fake.pdf", "page": 1})]


class FakeWebSearchTool:
    def invoke(self, args):
        return f"가짜 웹 검색 결과: {args['query']}"


def make_fake_graph(llm):
    return build_graph(llm=llm, llm_full=llm, tech_retriever=FakeRetriever(),
                       domain_retriever=FakeRetriever(),
                       web_search_tool=FakeWebSearchTool())


def run():
    result = make_fake_graph(FakeLLM()).invoke({}, config={"recursion_limit": 40})

    assert result["selected_technologies"]["SW"]["name"] == "TurboQuant", "A 노드"
    assert result["target_domain"] == "OnDevice AI"
    assert "TurboQuant" in result["tech_research"], "B 노드"
    assert result["market_eval"]["adoption_status"]["turboquant"] == "정식", "C - TechStatus 중첩"
    assert result["stakeholder_eval"]["label"] == "조건 의존", "D 노드"
    assert result["domain_eval"]["criteria"]["memory_budget"]["infinigen"].startswith("넘는다"), "E - 중첩"
    assert result["domain_eval"]["notes"], "E - notes 가 채워지는지"
    assert "conflicts" in result["synthesis"], "F 노드"
    assert "도메인" in result["final_report"], "G 가 H 피드백 받아 재생성했는지"
    assert result["validation_result"]["is_valid"] is True, "H 최종 통과"
    assert result["validation_result"]["forced_pass"] is False, "상한 도달 아님"
    assert result["retry_count"] == 1, f"H->G 재시도 1회, got {result['retry_count']}"

    # references 는 4개 키로 나뉘어 각자 채워진다. 개수 = 그 노드의 쿼리 수.
    # 이 단언이 깨지면 QUERY_TEMPLATES 가 바뀐 것이다 - 의도한 변경인지 볼 것.
    counts = {k: len(result[k]) for k in
              ["tech_references", "market_references",
               "stakeholder_references", "domain_references"]}
    assert counts == {"tech_references": 3, "market_references": 6,
                      "stakeholder_references": 8, "domain_references": 3}, counts

    print("배선 테스트 통과.")
    print(f"  - references: {counts} (총 {sum(counts.values())})")
    print(f"  - retry_count: {result['retry_count']}")
    print(f"  - validation_result: {result['validation_result']}")


class AlwaysBadReportLLM(FakeLLM):
    """G 가 몇 번을 다시 써도 계속 장이 빠진 보고서만 내는 경우.
    H 가 MAX_RETRY_H 를 넘기면 forced_pass=True 로 강제 종료하는지 본다."""

    def invoke(self, prompt):
        return FakeMessage("# SUMMARY\n내용만 있고 다른 장은 계속 빠짐")


def run_forced_pass_scenario():
    result = make_fake_graph(AlwaysBadReportLLM()).invoke({}, config={"recursion_limit": 40})

    v = result["validation_result"]
    assert v["forced_pass"] is True, "상한 도달 시 강제 통과"
    assert v["is_valid"] is True, "forced_pass 여도 is_valid=True 로 END 라우팅"
    assert result["retry_count"] == config.MAX_RETRY_H + 1, \
        f"최초 1회 + 재시도 {config.MAX_RETRY_H}회, got {result['retry_count']}"
    assert v["missing_items"], "missing_items 가 남아야 한다(한계점 장 기재용)"
    assert "보고서 검증 기록" in result["final_report"], "감사 노트가 보고서에 붙었는지"

    print("forced_pass 시나리오 통과.")
    print(f"  - retry_count: {result['retry_count']}")
    print(f"  - missing_items: {v['missing_items']}")


run()
print()
run_forced_pass_scenario()

## 7. 실제 실행

API 키가 있어야 여기서부터 의미가 있다. `recursion_limit`은 최악의 경우(H가 2번 재시도)를 감안해 넉넉히 잡는다.

In [ ]:
result = graph.invoke({}, config={"recursion_limit": 40})

print("=== validation_result ===")
print(result["validation_result"])
print("\n=== retry_count ===")
print(result["retry_count"])

## 8. 보고서 저장

In [ ]:
from pathlib import Path

output_dir = Path("../output")
output_dir.mkdir(exist_ok=True)
report_path = output_dir / "final_report.md"
report_path.write_text(result["final_report"], encoding="utf-8")

print(f"저장 완료: {report_path}")
print("\n--- 미리보기 (앞 1000자) ---")
print(result["final_report"][:1000])